# Results for model: institute-of-science-tokyo/llama-3.1-swallow-70b-instruct-v0.1

In [1]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Calculate global TOP_QUANTILE of 'feature_00' relative to 'responder_6'
df = df.with_columns([
    pl.col('feature_00').rank(method='dense').alias('feature_00_rank')
])
df = df.with_columns([
    pl.when(pl.col('feature_00_rank') > 0.85).then(1).otherwise(0).alias('TOP_QUANTILE')
])

# Convert 'date_id' to categorical
df = df.with_columns([
    pl.col('date_id').cast(pl.Categorical)
])

# Split data into training and testing sets
train_df, test_df = df.random_split(0.8, seed=42)

# Train XGBoost Regressor
X_train = train_df.drop(['responder_6']).to_pandas()
y_train = train_df['responder_6'].to_pandas()
X_test = test_df.drop(['responder_6']).to_pandas()
y_test = test_df['responder_6'].to_pandas()

xgb_model = xgb.XGBRegressor()
xgb_model.fit(X_train, y_train)

# Make predictions
y_pred = xgb_model.predict(X_test)

# Evaluate model
mse = ((y_test - y_pred) ** 2).mean()
print(f'MSE: {mse:.2f}')

FileNotFoundError: The system cannot find the path specified. (os error 3): ./jane_street_data/train.parquet